# Exercise 3: Streaming with the Responses API

Streaming delivers tokens to the client as they are generated, reducing  
perceived latency. This notebook shows how to consume streaming events,  
measure time-to-first-token (TTFT), and inspect the event types the API emits.

In [1]:
import time

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

## Stream a response and collect metrics

In [3]:
print("=== Streaming response ===\n")

start = time.time()
first_token_time = None
event_types_seen = set()

stream = client.responses.create(
    model="gpt-4.1-mini",
    input=(
        "Explain the difference between RAG and fine-tuning for enterprise AI "
        "deployments. Be concise -- 3 bullet points."
    ),
    stream=True,
)

print("Text: ", end="", flush=True)
for event in stream:
    event_types_seen.add(type(event).__name__)

    # ResponseTextDeltaEvent has the streaming text chunks
    if event.type == "response.output_text.delta":
        if first_token_time is None:
            first_token_time = time.time()
        print(event.delta, end="", flush=True)

    # ResponseCompletedEvent has the final response with usage
    if event.type == "response.completed":
        final_response = event.response

end = time.time()
print()  # newline after streamed text

=== Streaming response ===

Text: - **RAG (Retrieval-Augmented Generation):** Combines a retrieval system with a generative model to fetch relevant documents dynamically and generate responses, enabling up-to-date, context-aware outputs without model retraining.  
- **Fine-tuning:** Involves updating the weights of a pre-trained model on domain-specific datasets, resulting in a specialized model tailored for enterprise-specific language and tasks but requiring retraining and maintenance.  
- **Deployment impact:** RAG offers flexibility and scalability with easier updates to knowledge bases, while fine-tuning provides deeper domain adaptation at the cost of higher compute and update complexity.


## Streaming metrics

In [ ]:
print("=" * 60)
print("STREAMING METRICS")
print("=" * 60)
print(f"Time to first token: {(first_token_time - start)*1000:.0f}ms")
print(f"Total time:          {(end - start)*1000:.0f}ms")
print(f"Tokens (input):      {final_response.usage.input_tokens}")
print(f"Tokens (output):     {final_response.usage.output_tokens}")

## Event types observed

The stream emits several event types beyond just text deltas.

In [ ]:
print(f"Event types seen ({len(event_types_seen)}):")
for et in sorted(event_types_seen):
    print(f"  - {et}")

## Interview one-liner

> **Streaming** with `stream=True` lets you display tokens as they arrive, cutting perceived latency to the time-to-first-token (TTFT); the `response.completed` event at the end gives you the full response object with usage stats.